In [ ]:
# bootstrap: Colab clone + local import of `moecore` (auto-inserted)
import sys, pathlib
if "google.colab" in sys.modules:
    import os, subprocess
    _slug = "aniryou/full-stack-agentic-engineer"
    _repo = pathlib.Path("/content/full-stack-agentic-engineer")
    if not _repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_repo)], check=True)
    os.chdir(_repo / "00-foundations/mixture-of-experts/moe-core")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
_r = pathlib.Path.cwd().resolve()
while _r != _r.parent and not (_r / "moecore").exists():
    _r = _r.parent
if str(_r) not in sys.path:
    sys.path.insert(0, str(_r))
del _r

# 02 · Routing and load balance

**Tier:** T0 — numpy on a laptop or Colab CPU; the training runs take about 10 seconds in all. The torch version
of the same experiment is `../moe-lab/notebooks/01_a_tiny_moe_in_torch.ipynb` (T0 with torch installed).

## The one-minute version
A router is trained by the same loss as the experts, and that loss rewards sending a token to whichever expert
is *already* good at it. The expert that wins early gets the gradient, improves, and wins more; an expert that
gets no tokens never trains and never gets a chance. Left alone, a layer **collapses** onto a few experts — the
rest are dead weight in HBM. Balance has to be imposed: an **auxiliary loss** `E · Σ f_e · P_e` (Switch/GShard,
smallest when token shares f and mean probabilities P are both uniform); a **capacity** per expert with the
overflow dropped, or **dropless** routing with padding (MegaBlocks, and every inference engine); a
**selection-only bias** stepped against the load (DeepSeek-V3's auxiliary-loss-free balancing); or
**expert-choice** routing, where experts pick tokens. The **z-loss** keeps router logits small for bf16. After
this notebook you can compute each of these by hand, watch a router collapse and recover, and say what each fix
costs.

Primer: `../PRIMER.md` §3 *Routing and load balance* and §4 *Training MoE in brief*.

> **Exercise cells** contain `# YOUR CODE HERE` — replace it, then run the **Check** cell below it. A check prints ✅ when it passes. The finished version is in `solutions/`.

In [ ]:
import numpy as np

from moecore import routing as R
from moecore import train as T
from moecore.moe import ROUTERS, route, softmax

rng = np.random.default_rng(0)

## Worked example 1 — the Switch loss, in both normalisations
`f_e` is the share of the batch's assignments that went to expert e and `P_e` the mean router probability of e.
transformers' `load_balancing_loss_func` lets Σ f = k, so a perfectly uniform router scores **k** (2 for Mixtral);
Megatron and MegaBlocks divide by k, so uniform scores **1**. Both are E at worst (k = 1, one expert takes all).
Only P carries a gradient: f comes out of a top-k.

In [ ]:
t, e, k = 64, 8, 2
uniform_p = np.full((t, e), 1 / e)
spread = np.array([[(i * k + j) % e for j in range(k)] for i in range(t)])
lumped = np.tile([0, 1], (t, 1))
lumped_p = np.zeros((t, e)); lumped_p[:, :2] = 0.5
print(f"{'':28s} {'hf (Σf = k)':>12s} {'megatron (Σf = 1)':>18s}")
for name, p, idx in (("uniform", uniform_p, spread), ("two experts take everything", lumped_p, lumped)):
    print(f"{name:28s} {R.switch_aux_loss(p, idx, 'hf'):12.2f} {R.switch_aux_loss(p, idx, 'megatron'):18.2f}")
print(f"z-loss of all-zero logits over 8 experts: (ln 8)^2 = {R.z_loss(np.zeros((4, 8))):.3f}; "
      f"the same logits + 20: {R.z_loss(np.full((4, 8), 20.0)):.0f} (softmax unchanged, z-loss not)")

## Worked example 2 — capacity, dropping, and dropless padding
With a capacity factor, expert e takes at most `int(factor · k · T / E)` rows; the rest are **dropped** — those
tokens skip the expert and ride the residual connection. Dropless routing keeps every assignment and instead
pads each expert's rows to the kernel's block size: vLLM's `moe_align_block_size` sorts the T·k slots by expert
and pads each segment to a multiple of `BLOCK_SIZE_M` with a pad id (T·k), so no block GEMM mixes experts.

In [ ]:
logits = np.random.default_rng(1).standard_normal((256, 8)) + np.array([1.5, 0.8, 0, 0, 0, 0, 0, 0])  # 0, 1 popular
rr = route(logits, 2, **ROUTERS["mixtral"])
print("assignments per expert:", R.load(rr.idx, 8).tolist())
for factor in (1.0, 1.25, 2.0):
    cap = R.capacity(256, 8, 2, factor)
    kept = R.apply_capacity(rr.idx, rr.weights, cap)
    print(f"capacity factor {factor:4.2f}: {cap:3d} rows/expert, dropped {1 - kept.mean():6.1%} of assignments")
ids, blocks, padded = R.align_block_size(rr.idx, 16, 8)
print(f"dropless: {rr.idx.size} assignments padded to {padded} rows in {len(blocks)} blocks of 16 "
      f"({padded / rr.idx.size - 1:.1%} padding)")

## Worked example 3 — collapse, and two ways out
`moecore.train` trains a tiny MoE with hand-written gradients: four clusters of tokens, each with its own target
map; four linear experts; a linear top-1 router with Switch-style weights. Hidden states share a big common
direction (as transformer hidden states do), so at step 0 two experts top most tokens. Watch the share of
tokens per expert.

In [ ]:
task = T.make_task(seed=6)
runs = {b: T.train(task, balance=b) for b in ("none", "aux", "bias")}
for b, h in runs.items():
    print(f"\n{b}: task loss {h.loss[0]:.3f} -> {h.loss[-1]:.3f}")
    for step in (0, 25, 50, 100, 299):
        print(f"  step {step:3d}: share per expert {np.round(h.share[step], 2).tolist()}")
print("\nwhere each cluster's tokens went at the end (rows: clusters, cols: experts):")
for b in ("none", "aux", "bias"):
    print(f"{b:5s}", runs[b].placement.tolist())

Without balancing the router starts 55/45 on two experts and ends with one expert taking 95% — the other three
never learn anything. With the aux loss (α = 0.1) or the DeepSeek-style bias, all four experts carry about a
quarter and the task loss is three to five times lower. Look at the placements: the bias run gives each cluster
its own expert almost cleanly; the aux run balances too, but its gradient insists on equal counts while the
clusters are unequal (124, 117, 135, 136 tokens), so it splits two clusters across experts — the quality cost of
balancing through the loss. Across ten task seeds the pattern holds (a toy: the numbers depend on the seed, the
direction does not):

In [ ]:
print(f"{'seed':>4s} | {'none: experts used, loss':>26s} | {'aux':>18s} | {'bias':>18s}")
for seed in range(10):
    tk = T.make_task(seed=seed)
    cells = []
    for b in ("none", "aux", "bias"):
        h = T.train(tk, balance=b)
        cells.append(f"{(h.final_share > 0.05).sum()} used, {h.loss[-1]:.3f}")
    print(f"{seed:4d} | {cells[0]:>26s} | {cells[1]:>18s} | {cells[2]:>18s}")

## Exercise 2.1 — the Switch loss
Write `switch_loss(probs, idx)` in transformers' convention: `E · Σ_e f_e · P_e` with `f_e` = assignments to e
divided by T (so Σ f = k) and `P_e` = mean of `probs[:, e]`.

In [ ]:
def switch_loss(probs, idx):
    # YOUR CODE HERE
    raise NotImplementedError("your turn")

In [ ]:
for _ in range(5):
    p = softmax(rng.standard_normal((40, 16)))
    idx = np.argsort(-p, axis=1)[:, :4]
    assert np.isclose(switch_loss(p, idx), R.switch_aux_loss(p, idx, "hf"))
assert np.isclose(switch_loss(uniform_p, spread), 2.0)
print("✅ E·Σ f·P: k when uniform (hf), E when one expert takes all; divide by k for Megatron's convention")

## Exercise 2.2 — what does a capacity factor drop?
For the skewed batch `rr` above (256 tokens, 8 experts, top-2), set `drop_100` and `drop_125` to the fraction of
assignments dropped at capacity factor 1.0 and 1.25 (position policy), and `factor_for_zero` to the smallest
factor, in steps of 0.25, that drops nothing. Use `R.capacity` and `R.apply_capacity`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError("your turn")

In [ ]:
loads = R.load(rr.idx, 8)
assert np.isclose(drop_100, np.maximum(loads - 64, 0).sum() / 512)
assert drop_125 < drop_100 and dropped(factor_for_zero) == 0 and dropped(factor_for_zero - 0.25) > 0
assert R.capacity(256, 8, 2, factor_for_zero) >= loads.max()
print(f"✅ factor 1.0 drops {drop_100:.1%}, 1.25 drops {drop_125:.1%}; nothing is dropped until {factor_for_zero:.2f}, "
      f"the first step above the hottest expert's load over the mean ({loads.max()} / {loads.mean():.0f} = "
      f"{loads.max() / loads.mean():.2f}). Inference engines are dropless: a dropped token changes the answer")

## Exercise 2.3 — balance with a bias alone
Freeze a skewed router (`scores` below: experts 0 and 1 favoured) and balance it with DeepSeek's rule: select
with `scores + bias`, and after each batch step every expert's bias by `rate` toward the mean load —
`bias += rate · sign(total − load · E)`. Write `balance_by_bias(scores, k, rate, steps)` returning the final bias
and the load of the last step. The weights would still come from `scores`: the bias only chooses.

In [ ]:
scores = softmax(np.random.default_rng(2).standard_normal((512, 8)) + np.array([2.0, 1.0, 0, 0, 0, 0, 0, 0]))
print("load with no bias:", R.load(np.argsort(-scores, axis=1)[:, :2], 8).tolist())

In [ ]:
def balance_by_bias(scores, k, rate, steps):
    # YOUR CODE HERE
    raise NotImplementedError("your turn")

In [ ]:
bias, counts = balance_by_bias(scores, 2, 0.002, 400)
ref = np.zeros(8)
for _ in range(400):
    c = R.load(np.argsort(-(scores + ref), axis=1)[:, :2], 8)
    ref = R.update_bias(ref, c, 0.002)
assert np.allclose(bias, ref)
assert R.stats(counts)["max_over_mean"] < 1.2 and bias[0] < 0 < bias[-1]
print(f"✅ load after 400 bias steps: {counts.tolist()} (max/mean {R.stats(counts)['max_over_mean']:.2f}); "
      f"bias {np.round(bias, 3).tolist()} - no gradient touched the router")

## Exercise 2.4 — how strong must the aux loss be?
On task seed 4, train with the aux loss at α = 0.01 and α = 0.1. Before running, predict which one still
collapses; set `collapsed` to the α that does and `balanced` to the one that does not.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError("your turn")

In [ ]:
tk4 = T.make_task(seed=4)
assert T.train(tk4, balance="aux", alpha=collapsed).final_share.max() > 0.9
assert T.train(tk4, balance="aux", alpha=balanced).final_share.max() < 0.35
print("✅ α = 0.01 is too weak to pull tokens off the winning expert here; 0.1 spreads them. Too strong trades "
      "quality for balance (it forces equal counts even when the data is not balanced) - DeepSeek-V3's reason to move "
      "balancing out of the loss and into a bias")

## Exercise 2.5 — balanced on average, collapsed per sequence
Build `probs` and `idx` for 4 sequences of 16 tokens and 4 experts, top-1, in which **sequence i sends every
token to expert i** with probability 1. Then set `batch_loss` (Megatron convention over the whole batch) and
`seq_loss` (`R.sequence_aux_loss`).

In [ ]:
# YOUR CODE HERE
raise NotImplementedError("your turn")

In [ ]:
assert np.isclose(batch_loss, 1.0) and np.isclose(seq_loss, 4.0)
print("✅ the batch-level loss says perfect (1.0), the sequence-level one says collapsed (4.0 = E). DeepSeek-V3 keeps "
      "a small sequence-wise term for this; at inference, a long sequence that hammers one expert is a hot expert")

## In a design review
**The two-minute version.** "A learned router is a feedback loop: the expert that wins tokens early trains on
them and keeps winning, so without a counter-force the layer collapses onto a few experts and the rest are dead
HBM. Training adds one: the Switch auxiliary loss E·Σ f·P — minimised at k in transformers' convention and 1 in
Megatron's, so check which before comparing coefficients — plus a small z-loss for bf16 stability. Capacity
factors bound each expert's work by dropping overflow tokens, which is fine for training throughput and
unacceptable at inference, so engines are dropless and pad instead. DeepSeek-V3 moved balancing out of the loss:
a per-expert bias, stepped by a fixed rate against the load, changes which experts are chosen but never the
weights, so the task gradient stays clean. Balance at training time is on average; at inference, a skewed
workload still makes hot experts, and that is an expert-parallel problem."

**Drill questions**
1. *A paper says 'aux loss coefficient 0.01'; your framework's loss reads 2.0 on a uniform router. Why?* —
   transformers' convention counts all k assignments (Σ f = k); Megatron's divides by k. Same coefficient, a
   k-times different gradient.
2. *Why can't DeepSeek's bias just be added to the combine weights too?* — Then it would change every token's
   output and the task gradient would fight it; as a selection-only term it steers load without biasing outputs.
3. *Expert-choice routing is perfectly balanced. Why don't decode engines use it?* — Each expert picks from the
   whole batch, so a token's experts depend on the other tokens in the step and it can get zero experts; that is
   neither causal nor deterministic per request.